# Linear Regression — OLS Approach From Scratch

## What is OLS?

**Ordinary Least Squares (OLS)** is the mathematical method used to fit a straight line through data.

The line has the form:
$$
\hat{y} = mx + c
$$
where **m** is the slope and **c** is the intercept.

OLS finds the values of m and c that make the line as close as possible to all the data points at once.

## Why Squared Errors?

For each data point, the line makes an error:
$$
\text{error}_i = y_i - \hat{y}_i
$$

We cannot just sum the raw errors — positive and negative errors cancel each other out.
Instead, OLS minimises the **Sum of Squared Errors (SSE)**:
$$
\text{SSE} = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
$$

Squaring does two things:
- Removes the sign — positive and negative errors no longer cancel
- Punishes bigger errors more heavily — a mistake of 4 contributes 16, a mistake of 2 contributes only 4

## The Closed-Form Solution

OLS gives us exact formulas for the best slope and intercept — no iteration needed:

$$
m = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2}
$$

$$
c = \bar{y} - m \cdot \bar{x}
$$

Where $\bar{x}$ and $\bar{y}$ are the means of x and y.

**Intuition for the slope formula:**
- The numerator measures how x and y move together (covariance)
- The denominator measures how much x varies (variance of x)
- The ratio tells us: for every 1 unit increase in x, how much does y change?

## Step 1 — Dataset

We use hours studied vs exam marks with realistic noise.
The relationship is roughly linear but not perfect — just like real data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

hours  = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=float)
# True relationship: marks = 8*hours + 20, plus noise
marks  = 8 * hours + 20 + np.random.normal(0, 5, len(hours))

df = pd.DataFrame({'x': hours, 'y': marks})
print(df.to_string(index=False))

plt.figure(figsize=(7, 4))
plt.scatter(df['x'], df['y'], color='steelblue', s=80, zorder=5)
plt.title('Hours Studied vs Exam Marks')
plt.xlabel('Hours studied (x)')
plt.ylabel('Exam marks (y)')
plt.grid(True)
plt.tight_layout()
plt.show()

## Step 2 — Compute Slope and Intercept Step by Step

We build the OLS calculation inside a DataFrame so every intermediate value is visible.
This makes it easy to trace exactly where the slope formula comes from.

In [ ]:
df['x_mean'] = df['x'].mean()
df['y_mean'] = df['y'].mean()
df['x - x_mean']             = df['x'] - df['x_mean']
df['y - y_mean']             = df['y'] - df['y_mean']
df['(x-xm)(y-ym)']           = df['x - x_mean'] * df['y - y_mean']
df['(x-xm)^2']               = df['x - x_mean'] ** 2

print(df[['x', 'y', 'x - x_mean', 'y - y_mean', '(x-xm)(y-ym)', '(x-xm)^2']].round(2).to_string(index=False))

In [ ]:
numerator   = df['(x-xm)(y-ym)'].sum()
denominator = df['(x-xm)^2'].sum()

slope     = numerator / denominator
intercept = df['y'].mean() - slope * df['x'].mean()

print(f'Numerator   sum((x-xm)(y-ym)) = {numerator:.4f}')
print(f'Denominator sum((x-xm)^2)     = {denominator:.4f}')
print()
print(f'Slope     m = {numerator:.4f} / {denominator:.4f} = {slope:.4f}')
print(f'Intercept c = y_mean - m*x_mean = {df["y"].mean():.4f} - {slope:.4f}*{df["x"].mean():.4f} = {intercept:.4f}')
print()
print(f'Regression line: y_hat = {slope:.4f} * x + {intercept:.4f}')

## Step 3 — Plot the Regression Line

We overlay the fitted line on the scatter plot.
The line minimises the total squared distance from all points to the line.

In [ ]:
x_line  = np.linspace(df['x'].min() - 0.5, df['x'].max() + 0.5, 200)
y_line  = slope * x_line + intercept

# Also compute residuals (errors)
y_hat   = slope * df['x'] + intercept
residuals = df['y'] - y_hat

plt.figure(figsize=(8, 5))
plt.scatter(df['x'], df['y'], color='steelblue', s=80, zorder=5, label='Data points')
plt.plot(x_line, y_line, color='tomato', linewidth=2, label=f'y = {slope:.2f}x + {intercept:.2f}')

# Draw residual lines
for xi, yi, yhi in zip(df['x'], df['y'], y_hat):
    plt.plot([xi, xi], [yi, yhi], color='gray', linewidth=0.8, linestyle='--')

plt.title('OLS Regression Line — Minimises Total Squared Residuals')
plt.xlabel('Hours studied')
plt.ylabel('Exam marks')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

sse = (residuals ** 2).sum()
print(f'Sum of Squared Errors (SSE) = {sse:.4f}')
print(f'This is the minimum possible SSE for any straight line through this data.')

## Step 4 — Predictions

With slope and intercept known, predicting is simple:
plug any new x value into `y_hat = m*x + c`.

In [ ]:
df['y_hat'] = slope * df['x'] + intercept
df['residual'] = df['y'] - df['y_hat']

print(df[['x', 'y', 'y_hat', 'residual']].round(2).to_string(index=False))
print()

# Predict new values
for x_new in [12, 15, 20]:
    y_new = slope * x_new + intercept
    print(f'  x = {x_new} hours  -->  predicted marks = {y_new:.1f}')

## Step 5 — Reusable OLS Function

We wrap the calculation into a clean function that works on any x and y arrays.

In [ ]:
def ols(x, y):
    """Compute OLS slope and intercept for 1D arrays x and y."""
    x_mean = x.mean()
    y_mean = y.mean()
    slope     = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean) ** 2)
    intercept = y_mean - slope * x_mean
    return slope, intercept

def predict(x, slope, intercept):
    return slope * x + intercept

# Verify on our dataset
m, c = ols(df['x'].values, df['y'].values)
print(f'slope = {m:.4f},  intercept = {c:.4f}')
print(f'Prediction at x=12: {predict(12, m, c):.2f}')

## Step 6 — Test on a Different Dataset

We generate a new dataset with a different true relationship
to show the function works generally — not just for our specific example.

In [ ]:
np.random.seed(7)
x2 = np.linspace(0, 50, 30)
y2 = 3.5 * x2 + 10 + np.random.normal(0, 15, 30)   # true: y = 3.5x + 10

m2, c2 = ols(x2, y2)
print(f'True relationship:   y = 3.5x + 10')
print(f'OLS recovered:       y = {m2:.4f}x + {c2:.4f}')

plt.figure(figsize=(7, 4))
plt.scatter(x2, y2, color='steelblue', s=50, label='Data', zorder=5)
plt.plot(x2, predict(x2, m2, c2), color='tomato', linewidth=2,
         label=f'OLS fit: y = {m2:.2f}x + {c2:.2f}')
plt.title('OLS on a Different Dataset')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Step 7 — Compare With sklearn

sklearn's `LinearRegression` uses the same OLS formulas internally.
Our slope and intercept should match exactly.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

sk_model = LinearRegression()
sk_model.fit(df[['x']], df['y'])

sk_slope     = sk_model.coef_[0]
sk_intercept = sk_model.intercept_

print(f'Our OLS:  slope = {slope:.6f},  intercept = {intercept:.6f}')
print(f'sklearn:  slope = {sk_slope:.6f},  intercept = {sk_intercept:.6f}')
print()

y_hat_sk = sk_model.predict(df[['x']])
print(f'R2 score:  {r2_score(df["y"], df["y_hat"]):.6f}')
print(f'MSE:       {mean_squared_error(df["y"], df["y_hat"]):.6f}')

## Step 8 — What R² Means

R² (R-squared) tells us how much of the variation in y is explained by x.

$$
R^2 = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2} = 1 - \frac{SSE}{SST}
$$

- **R² = 1.0** — perfect fit, model explains all variation
- **R² = 0.0** — model is no better than predicting the mean
- **R² = 0.85** — model explains 85% of the variation in y

In [ ]:
SSE = ((df['y'] - df['y_hat']) ** 2).sum()               # error from our line
SST = ((df['y'] - df['y'].mean()) ** 2).sum()            # error from the mean

r2 = 1 - SSE / SST

print(f'SSE (our line):   {SSE:.4f}')
print(f'SST (mean line):  {SST:.4f}')
print(f'R2 = 1 - SSE/SST = 1 - {SSE:.4f}/{SST:.4f} = {r2:.4f}')
print(f'\nOur model explains {r2*100:.1f}% of the variation in exam marks.')

## Summary

| Step | Formula | What it computes |
|------|---------|------------------|
| Mean | x_bar, y_bar | Centre of the data |
| Numerator | sum((x - x_bar)(y - y_bar)) | How x and y move together |
| Denominator | sum((x - x_bar)^2) | How much x varies |
| Slope | numerator / denominator | Change in y per unit change in x |
| Intercept | y_bar - m * x_bar | Where the line crosses y-axis |
| Prediction | m*x + c | Estimated y for any new x |
| R² | 1 - SSE/SST | Fraction of variation explained |

## OLS vs Gradient Descent

| Scenario | OLS | Gradient Descent |
|----------|-----|------------------|
| Linear regression (small data) | Best choice — exact, fast, no tuning | Works but unnecessary |
| Large datasets (millions of rows) | Slow — matrix inversion is expensive | Fast — processes mini-batches |
| Logistic regression | Cannot use — no closed form | Required |
| Neural networks | Cannot use — no closed form | Only option |
| Regularisation (Ridge, Lasso) | Hard or impossible | Works naturally |

## Key Insight

> OLS gives you the **mathematically perfect answer** in one shot.
> Gradient Descent approximates it over many steps.
> For linear regression on small/medium data, always prefer OLS.
> For everything else in ML, Gradient Descent is the standard.